# 07a — Final comment classification & analysis dataset

**Study:** *Likes or Dislikes, Gratifications or Concerns?* (R&R, Cyberpsychology)

**Was dieses Notebook macht:**

1. Lädt Survey-Daten (`data.csv`, N = 559) und die drei Forum-Exporte (VM1/VM2/VM3)
2. **Verknüpft Forum-Username → Survey-`id`** (validiert, 235/235 Poster)
3. Baut das **Comment-Level-Datenset** (1'211 echte Kommentare, davon 651 mit Survey-ID)
4. Klassifiziert `pers_exp`, `emot_exp`, `pol_opin`, `breadth`, `contr` per LLM (v5-Prompt)
5. Klassifiziert `valence` per **GerVADER** (Lexikon-Ansatz), kalibriert auf Ground Truth
6. Aggregiert auf **Person-Level inkl. der 324 Nicht-Kommentierenden** (Hurdle-Logik)
7. Exportiert die finalen Datensätze für die Auswertung

**Wichtig zur Struktur:** Es gibt zwei finale Datensätze, nicht einen. Kommentar-Inhalt existiert
nur für Leute, die kommentiert haben — Nicht-Kommentierende bekommen bei Inhaltsvariablen `NaN`,
nicht `0`. Details in Abschnitt 6.

## 0. Konfiguration

In [ ]:
from pathlib import Path
import re, json, warnings
import numpy as np
import pandas as pd

# ---- Pfade anpassen ----------------------------------------------------------
DATA   = Path("data")          # data.csv, data_posts_VM1-3.csv
OUT    = Path("output")        # Exporte dieses Notebooks
GT     = Path("data/comments_manual_coding.xlsx")   # 150 manuell codierte Kommentare
SPLIT  = Path("data/split_seed42.csv")              # few-shot pool vs. held-out (aus 04c)
PROMPT = Path("prompts/system_prompt_v5.txt")       # finaler System-Prompt
GERVADER_LEX = Path("lexicons/GERVaderLexicon.txt") # s. Abschnitt 5

# ---- LLM ---------------------------------------------------------------------
MODEL      = "qwen3:8b"     # finales Modell aus 04c
LLM_VARS   = ["pers_exp", "emot_exp", "pol_opin", "breadth", "contr"]
VADER_VARS = ["valence"]
ALL_VARS   = LLM_VARS + VADER_VARS

BREADTH_MAX = 3             # Cap laut Codebook v5

VM2VERSION = {"VM1": "Control", "VM2": "Like", "VM3": "Like & dislike"}

OUT.mkdir(exist_ok=True, parents=True)
pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", category=FutureWarning)
print("ok")

## 1. Daten laden und aufbereiten

Die Post-Exporte sind **cp1252**-kodiert (nicht UTF-8) und enthalten zwei verschiedene
Datumsformate. Beides wird hier abgefangen.

In [ ]:
def parse_dates(s: pd.Series) -> pd.Series:
    """Post-Timestamps: ISO mit Mikrosekunden ODER deutsches Format (minutengenau)."""
    a = pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S.%f", errors="coerce")
    b = pd.to_datetime(s, format="%d.%m.%Y %H:%M",       errors="coerce")
    c = pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S",    errors="coerce")
    return a.fillna(b).fillna(c)


def n_tokens(text) -> int:
    """Quanteda-nahe Tokenisierung (Wörter + Satzzeichen als eigene Tokens)."""
    return len(re.findall(r"[\w]+(?:['-][\w]+)*|[^\w\s]", str(text)))


# ---- Survey ------------------------------------------------------------------
survey = pd.read_csv(DATA / "data.csv")
survey["last_posted"] = pd.to_datetime(
    survey["last_posted_at"].str.replace(" UTC", "", regex=False), errors="coerce"
)

# ---- Forum-Posts -------------------------------------------------------------
posts = []
for vm, version in VM2VERSION.items():
    p = pd.read_csv(DATA / f"data_posts_{vm}.csv", encoding="cp1252")
    p["vm"], p["version"] = vm, version
    p["dt"] = parse_dates(p["created_at"])
    p["coarse_ts"] = pd.to_datetime(p["created_at"], format="%d.%m.%Y %H:%M",
                                    errors="coerce").notna()
    posts.append(p)
posts = pd.concat(posts, ignore_index=True)

# Exakte Dubletten entfernen (1 Fall: ralf_md, VM3, post_number 38)
n_before = len(posts)
posts = posts.drop_duplicates(subset=["vm", "user", "created_at", "raw"]).reset_index(drop=True)
print(f"Dubletten entfernt: {n_before - len(posts)}")

# ---- Seed-Posts markieren ----------------------------------------------------
# Alle vom System vorbefüllten Posts stammen aus JULI 2017; diese Usernames posten
# ausschliesslich im Juli. Die Trennung ist damit vollständig eindeutig.
posts["is_seed_post"] = posts["dt"].dt.month == 7
seed_users = set(posts.loc[posts["is_seed_post"], "user"])
posts["is_seed_user"] = posts["user"].isin(seed_users)

assert (posts["is_seed_user"] & ~posts["is_seed_post"]).sum() == 0, \
    "Seed-User hat ausserhalb Juli gepostet — Annahme prüfen!"

print(f"Posts gesamt: {len(posts)} | Seeds: {posts['is_seed_post'].sum()} | "
      f"echt: {(~posts['is_seed_post']).sum()}")
print(f"Survey: N = {len(survey)} | davon mit post_count > 0: {(survey['post_count'] > 0).sum()}")

## 2. Verknüpfung Username ↔ Survey-`id`

**Das Problem:** `data.csv` enthält die numerische Discourse-`id`, die Post-Exporte nur den
`user`-Namen. Keine gemeinsame Spalte.

**Die Brücke:** Der Discourse-User-Export in `data.csv` enthält `last_posted_at` — den
sekundengenauen Zeitstempel des letzten Posts. Das ist praktisch ein Fingerabdruck.

| Pass | Kriterium | Treffer |
|---|---|---|
| 1 | `last_posted_at` exakt auf die Sekunde, beidseitig eindeutig | 230 |
| 2 | `post_count` + Wortzahl (für Fälle mit nur minutengenauem Timestamp) | 4 |
| 3 | Timestamp ±90 s **und** `post_count` identisch | 1 |

**Validierung:** Für alle 235 gematchten Personen stimmt anschliessend die Zahl der Posts im
Forum exakt mit `post_count` aus der Survey überein. Die Asserts unten brechen ab, falls das
nicht mehr der Fall ist.

⚠️ Der Schlüssel ist **`(vm, username)`**, nicht `username` allein: `anonym6`, `anonym8`,
`anonym9`, `anonym11` und `anonym14` kommen in mehreren VMs vor und sind verschiedene Personen.

In [ ]:
def link_users(survey: pd.DataFrame, posts: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for vm, version in VM2VERSION.items():
        real = posts[(posts["vm"] == vm) & (~posts["is_seed_user"])]
        agg = (real.groupby("user")
                   .agg(n_posts=("raw", "size"),
                        last=("dt", "max"),
                        words_est=("raw", lambda s: sum(n_tokens(x) for x in s)))
                   .reset_index())
        agg["last_s"] = agg["last"].dt.floor("s")

        sub = survey[(survey["version"] == version) & (survey["post_count"] > 0)].copy()
        sub["lp_s"] = sub["last_posted"].dt.floor("s")

        taken, matched = set(), {}

        # Pass 1: exakter Sekunden-Timestamp, beidseitig eindeutig
        counts = agg["last_s"].value_counts()
        lut = {t: u for t, u in zip(agg["last_s"], agg["user"]) if counts[t] == 1}
        for _, r in sub.iterrows():
            u = lut.get(r["lp_s"])
            if u is not None and u not in taken:
                taken.add(u); matched[r["id"]] = (u, "timestamp_exact")

        rest = agg[~agg["user"].isin(taken)]
        for _, r in sub[~sub["id"].isin(matched)].iterrows():
            # Pass 2: post_count + Wortzahl (Toleranz für Tokenisierungsunterschiede)
            c = rest[(rest["n_posts"] == r["post_count"]) &
                     (abs(rest["words_est"] - r["words"]) <= max(3, 0.08 * r["words"])) &
                     (~rest["user"].isin(taken))]
            if len(c) == 1:
                u = c["user"].iloc[0]
                taken.add(u); matched[r["id"]] = (u, "fingerprint_n_words"); continue
            # Pass 3: Timestamp ±90s UND post_count identisch
            c = rest[(abs((rest["last"] - r["last_posted"]).dt.total_seconds()) <= 90) &
                     (rest["n_posts"] == r["post_count"]) & (~rest["user"].isin(taken))]
            if len(c) == 1:
                u = c["user"].iloc[0]
                taken.add(u); matched[r["id"]] = (u, "timestamp_min_plus_n")
            else:
                matched[r["id"]] = (None, f"UNRESOLVED({len(c)})")

        for sid, (u, how) in matched.items():
            rows.append({"id": sid, "vm": vm, "version": version,
                         "username": u, "match_method": how})
    return pd.DataFrame(rows)


mapping = link_users(survey, posts)

# ---- Validierung -------------------------------------------------------------
n_posters = (survey["post_count"] > 0).sum()
n_matched = mapping["username"].notna().sum()
print(f"Poster in Survey: {n_posters} | gematcht: {n_matched}")
print(mapping["match_method"].value_counts().to_dict())

assert n_matched == n_posters, "Nicht alle Poster konnten gematcht werden!"
assert mapping.groupby(["vm", "username"]).size().max() == 1, "Username doppelt vergeben!"

mapping.to_csv(OUT / "mapping_id_username.csv", index=False)

In [ ]:
# Harte Validierung: stimmt die Zahl der verknüpften Posts mit post_count überein?
lut = {(vm, u): i for vm, u, i in zip(mapping["vm"], mapping["username"], mapping["id"])}
posts["id"] = [lut.get((vm, u)) for vm, u in zip(posts["vm"], posts["user"])]

linked_counts = (posts[~posts["is_seed_post"] & posts["id"].notna()]
                 .groupby("id").size().rename("n_linked"))
check = survey.set_index("id").loc[linked_counts.index, ["post_count", "words"]].join(linked_counts)
mismatch = check[check["n_linked"] != check["post_count"]]

print(f"Personen mit verknüpften Kommentaren: {len(check)}")
print(f"Abweichungen post_count vs. verknüpfte Posts: {len(mismatch)}")
assert len(mismatch) == 0, mismatch
print("✓ Verknüpfung validiert")

## 3. Comment-Level-Datenset

Alle 1'211 echten Kommentare. `id` ist `NaN` für Kommentare von Accounts, die im Paper nie
im Analysesample waren (960 erstellte Accounts vs. N = 559 über T1/T2-Token matchbar).

- **651 Kommentare von 235 Personen** → verknüpfbar mit Survey-Variablen
- **560 Kommentare** → nur für deskriptive Inhaltsanalysen (Bedingung ist trotzdem bekannt)

In [ ]:
comments = posts[~posts["is_seed_post"]].copy()
comments["text"] = (comments["raw"].fillna("")
                    .str.replace(r"<[^>]+>", " ", regex=True)     # HTML raus
                    .str.replace(r"\s+", " ", regex=True).str.strip())
comments["n_words"] = comments["text"].map(n_tokens)
comments["linked"] = comments["id"].notna()
comments["comment_id"] = (comments["vm"] + "_" +
                          comments["post_number"].astype("Int64").astype(str))

# Reaktionsspalten (Likes/Dislikes) je nach VM unterschiedlich benannt
react_cols = [c for c in comments.columns if c.startswith(("key", "value"))]

comments = comments[["comment_id", "vm", "version", "id", "username", "linked",
                     "post_number", "topic", "dt", "reply_to_post_number",
                     "text", "n_words"] + react_cols].sort_values(["vm", "post_number"])

comments.to_csv(OUT / "comments_level.csv", index=False, encoding="utf-8-sig")
print(f"Kommentare: {len(comments)} | mit Survey-ID: {comments['linked'].sum()} "
      f"| von {comments['id'].nunique()} Personen")
comments.head(3)

## 4. LLM-Klassifikation: `pers_exp`, `emot_exp`, `pol_opin`, `breadth`, `contr`

Übernimmt den Setup aus 04c. Die Schleife **speichert nach jedem Batch auf Platte** und ist
wiederaufnehmbar — ein Kernel-Verlust kostet damit maximal einen Batch (das war in früheren
Sessions das Hauptproblem).

`valence` wird hier bewusst **nicht** vom LLM codiert (κ = .38, unter Ziel) — s. Abschnitt 5.

In [ ]:
import ollama

SYSTEM_PROMPT = PROMPT.read_text(encoding="utf-8")
print(f"Prompt geladen: {len(SYSTEM_PROMPT)} Zeichen")

# Sanity-Check, dass wirklich v5 geladen ist (in 04c war v4 das Problem)
assert "HOW TO DECIDE" in SYSTEM_PROMPT, "Falscher Prompt — v5 erwartet!"

# Few-Shot-Turns aus dem Trainingspool (nicht aus dem Held-out-Set!)
gt    = pd.read_excel(GT)
split = pd.read_csv(SPLIT)                       # Spalten: comment_id/index, split
gt    = gt.merge(split, on="comment_id", how="left") if "comment_id" in split.columns else gt

def build_fewshot(pool: pd.DataFrame) -> list:
    msgs = []
    for _, r in pool.iterrows():
        msgs.append({"role": "user", "content": r["raw"]})
        msgs.append({"role": "assistant",
                     "content": json.dumps({v: int(r[v]) for v in LLM_VARS},
                                           ensure_ascii=False)})
    return msgs

FEWSHOT = build_fewshot(gt[gt["split"] == "train"]) if "split" in gt.columns else []
print(f"Few-Shot-Turns: {len(FEWSHOT)}")

In [ ]:
BOUNDS = {"pers_exp": (0, 1), "emot_exp": (0, 1), "pol_opin": (0, 1),
          "breadth": (0, BREADTH_MAX), "contr": (0, 1)}


def classify_one(text: str) -> dict:
    resp = ollama.chat(
        model=MODEL,
        messages=[{"role": "system", "content": SYSTEM_PROMPT}]
                 + FEWSHOT
                 + [{"role": "user", "content": text}],
        format="json",
        think=False,                       # verhindert qwen3-Endlosschleife
        options={"temperature": 0, "num_predict": 256, "seed": 42},
    )
    out = json.loads(resp["message"]["content"])
    clean = {}
    for v, (lo, hi) in BOUNDS.items():
        try:
            clean[v] = int(np.clip(int(out[v]), lo, hi))
        except (KeyError, TypeError, ValueError):
            clean[v] = np.nan           # Parse-Fehler explizit als NaN, nicht als 0
    return clean


RESULTS = OUT / "llm_codes_final.csv"
todo = comments[["comment_id", "text"]].copy()

if RESULTS.exists():
    done = pd.read_csv(RESULTS)
    todo = todo[~todo["comment_id"].isin(done["comment_id"])]
    print(f"Fortsetzung: {len(done)} erledigt, {len(todo)} offen")
else:
    done = pd.DataFrame()

BATCH, buf = 25, []
for k, (_, r) in enumerate(todo.iterrows(), 1):
    rec = {"comment_id": r["comment_id"]}
    try:
        rec.update(classify_one(r["text"]))
    except Exception as e:
        rec.update({v: np.nan for v in LLM_VARS}); rec["error"] = str(e)[:120]
    buf.append(rec)

    if len(buf) >= BATCH or k == len(todo):                 # <-- Speichern IM Loop
        pd.DataFrame(buf).to_csv(RESULTS, mode="a", index=False,
                                 header=not RESULTS.exists())
        print(f"  {k}/{len(todo)} gespeichert", end="\r")
        buf = []

llm_codes = pd.read_csv(RESULTS).drop_duplicates(subset="comment_id", keep="last")
print(f"\nKlassifiziert: {len(llm_codes)}")
print("Parse-Fehler pro Variable:", llm_codes[LLM_VARS].isna().sum().to_dict())

## 5. `valence` via GerVADER

**Wichtig:** Das englische VADER-Lexikon funktioniert auf deutschem Text **nicht** — es findet
fast nichts und was es findet, ist Rauschen. Getestet:

```
Ich finde den 9-Punkte-Plan sinnvoll...   → compound = -0.60   (falsches Vorzeichen!)
Heisse Luft, wie Alles, was von Merkel... → compound =  0.00   (nichts erkannt)
```

Wir brauchen deshalb **GerVADER** (Tymann et al. 2019): dieselbe VADER-Mechanik (Negation,
Intensivierer, Grossschreibung, Satzzeichen), aber mit einem deutschen Lexikon aus SentiWS +
GermanPolarityClues, ~35'000 Einträge.

```bash
git clone https://github.com/KarstenAMF/GerVADER.git lexicons/GerVADER
```

**Mapping auf die 4 Codebook-Kategorien.** `valence` ist im Codebook keine Skala, sondern vier
Klassen — und der Compound-Score allein kann *ambivalent* (3) und *indifferent* (4) nicht
trennen. Beide liegen nahe 0. Die Lösung nutzt die Komponenten `pos` und `neg`:

| Code | Regel |
|---|---|
| 3 ambivalent | `pos` **und** `neg` beide über `MIX_MIN` → beide Pole vorhanden |
| 1 positiv | `compound ≥ POS_T` |
| 2 negativ | `compound ≤ NEG_T` |
| 4 indifferent | Rest (kein nennenswertes Sentiment) |

Die drei Schwellen werden auf dem **Trainingspool** optimiert und auf dem **Held-out-Set**
berichtet — sonst ist die Reliabilität nach oben verzerrt.

In [ ]:
import sys
sys.path.insert(0, str(GERVADER_LEX.parent))     # Ordner mit vaderSentimentGER.py
from vaderSentimentGER import SentimentIntensityAnalyzer

gervader = SentimentIntensityAnalyzer(str(GERVADER_LEX))

def vader_scores(text: str) -> dict:
    s = gervader.polarity_scores(str(text))
    return {"v_compound": s["compound"], "v_pos": s["pos"],
            "v_neg": s["neg"], "v_neu": s["neu"]}

# Kurztest an den Codebook-Beispielen
for t in ["Ich finde den 9-Punkte-Plan sinnvoll, um die Bevölkerung mehr zu schützen.",
          "Neun Punkte BlaBla, nichts konkretes, ein bisschen Gesülze.",
          "Im Prinzip ist der Plan ein guter Anfang, allein mir fehlt der Glaube an der Umsetzung.",
          "Sehe ich auch so."]:
    print(vader_scores(t), "|", t[:60])

In [ ]:
def to_valence(row, pos_t, neg_t, mix_min) -> int:
    if row["v_pos"] >= mix_min and row["v_neg"] >= mix_min:
        return 3                                   # ambivalent: beide Pole vorhanden
    if row["v_compound"] >= pos_t:
        return 1
    if row["v_compound"] <= neg_t:
        return 2
    return 4                                       # indifferent


from sklearn.metrics import cohen_kappa_score, accuracy_score
from itertools import product

gt_sc = gt.join(gt["raw"].apply(lambda t: pd.Series(vader_scores(t))))
train = gt_sc[gt_sc["split"] == "train"] if "split" in gt_sc.columns else gt_sc
test  = gt_sc[gt_sc["split"] == "test"]  if "split" in gt_sc.columns else gt_sc

grid = list(product(np.arange(0.05, 0.55, 0.05),      # POS_T
                    np.arange(-0.55, -0.04, 0.05),    # NEG_T
                    np.arange(0.05, 0.30, 0.05)))     # MIX_MIN

best = max(grid, key=lambda g: cohen_kappa_score(
    train["valence"], train.apply(to_valence, axis=1, args=g)))
POS_T, NEG_T, MIX_MIN = best
print(f"Beste Schwellen (train): POS={POS_T:.2f} NEG={NEG_T:.2f} MIX={MIX_MIN:.2f}")

pred = test.apply(to_valence, axis=1, args=best)
kappa = cohen_kappa_score(test["valence"], pred)
print(f"HELD-OUT: accuracy = {accuracy_score(test['valence'], pred):.3f} | "
      f"Cohen's κ = {kappa:.3f}   (Ziel: acc ≥ .80, κ ≥ .60)")
print(pd.crosstab(test["valence"], pred, rownames=["human"], colnames=["gervader"]))

if kappa < 0.60:
    print("\n⚠️  Ziel verfehlt — s. Plan B unten, bevor auf 1'211 Kommentare skaliert wird.")

In [ ]:
# Auf alle Kommentare anwenden
v = comments["text"].apply(lambda t: pd.Series(vader_scores(t)))
v["valence"] = v.apply(to_valence, axis=1, args=(POS_T, NEG_T, MIX_MIN))
v["comment_id"] = comments["comment_id"].values

coded = (comments
         .merge(llm_codes[["comment_id"] + LLM_VARS], on="comment_id", how="left")
         .merge(v, on="comment_id", how="left"))

# Depth-Score laut Codebook: Summe der drei Facetten (0–3)
coded["depth"] = coded[["pers_exp", "emot_exp", "pol_opin"]].sum(axis=1, min_count=3)

coded.to_csv(OUT / "comments_classified_final.csv", index=False, encoding="utf-8-sig")
print(coded[ALL_VARS + ["depth"]].describe().round(2))

## 6. Person-Level-Aggregation — inkl. der Nicht-Kommentierenden

Die 324 Personen ohne Kommentar **müssen** im Datensatz bleiben, sonst analysiert man nur die
Selbstselektion der Aktiven. Aber: bei Inhaltsvariablen gilt `NaN`, **nicht `0`**.

> Wer nichts geschrieben hat, hat nicht „Selbstenthüllungstiefe 0" — die Variable ist für ihn
> schlicht undefiniert. Eine 0 würde den Mittelwert nach unten ziehen und einen Effekt vortäuschen.

Das passt zur Hurdle-Logik des Papers und trennt die Analyse sauber in zwei Stufen:

| Stufe | Frage | N |
|---|---|---|
| 1 — Hurdle | Kommentiert jemand überhaupt? (`commented` 0/1) | **559** |
| 2 — Inhalt | *Was* schreibt jemand, gegeben er schreibt? | **235** Personen / 651 Kommentare |

Für Reviewer #1 („nicht nur *wie viel*, sondern *was*") ist Stufe 2 die neue Analyse; Stufe 1
repliziert die bestehenden Ergebnisse.

In [ ]:
CONTENT_AGG = {
    "n_comments":   ("comment_id", "size"),
    "words_total":  ("n_words", "sum"),
    "depth_mean":   ("depth", "mean"),
    "depth_max":    ("depth", "max"),
    "pers_exp_any": ("pers_exp", "max"),
    "emot_exp_any": ("emot_exp", "max"),
    "pol_opin_any": ("pol_opin", "max"),
    "pol_opin_prop":("pol_opin", "mean"),
    "breadth_mean": ("breadth", "mean"),
    "breadth_max":  ("breadth", "max"),
    "contr_any":    ("contr", "max"),
    "contr_prop":   ("contr", "mean"),
    "valence_mean_compound": ("v_compound", "mean"),
}

person = (coded[coded["linked"]]
          .groupby("id").agg(**CONTENT_AGG).reset_index())

# Modale Valenz-Kategorie pro Person (Mittelwert über nominale Codes wäre sinnlos)
vmode = (coded[coded["linked"]].groupby("id")["valence"]
         .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else np.nan)
         .rename("valence_mode").reset_index())
person = person.merge(vmode, on="id", how="left")

# ---- Merge auf ALLE 559 ------------------------------------------------------
final = survey.merge(person, on="id", how="left")
final["commented"] = (final["post_count"] > 0).astype(int)
final["n_comments"] = final["n_comments"].fillna(0).astype(int)   # Anzahl: 0 ist korrekt
# Inhaltsvariablen bleiben bewusst NaN für Nicht-Kommentierende!

content_cols = [c for c in CONTENT_AGG if c != "n_comments"] + ["valence_mode"]
assert final.loc[final["commented"] == 0, content_cols].isna().all().all(), \
    "Nicht-Kommentierende haben Inhaltswerte — Aggregation prüfen!"

print(f"Final person-level: N = {len(final)}")
print(final.groupby(["version", "commented"]).size().unstack(fill_value=0))
print("\nInhaltsvariablen vorhanden für:", final["depth_mean"].notna().sum(), "Personen")

In [ ]:
# ---- Exporte -----------------------------------------------------------------
final.to_csv(OUT / "final_person_level.csv", index=False, encoding="utf-8-sig")

analysis_comments = coded[coded["linked"]].merge(
    survey[["id", "version", "age", "male", "edu",
            "pri_con_fs", "grats_gen_fs", "pri_del_fs", "self_eff_fs",
            "trust_gen_fs", "trust_spec_fs"]],
    on="id", how="left", suffixes=("", "_srv"))
analysis_comments.to_csv(OUT / "final_comment_level.csv", index=False, encoding="utf-8-sig")

print("Geschrieben:")
for f in ["final_person_level.csv", "final_comment_level.csv",
          "comments_classified_final.csv", "mapping_id_username.csv"]:
    print(f"  {OUT / f}")

print(f"\nPerson-Level : N = {len(final)} (davon {final['commented'].sum()} Kommentierende)")
print(f"Comment-Level: N = {len(analysis_comments)} Kommentare von "
      f"{analysis_comments['id'].nunique()} Personen")

## 7. Sanity-Checks vor der Auswertung

In [ ]:
checks = {
    "Survey-N = 559":                       len(final) == 559,
    "Alle Poster gematcht (235)":           final["commented"].sum() == 235,
    "Kommentare gesamt = 1211":             len(comments) == 1211,
    "Verknüpfte Kommentare = 651":          comments["linked"].sum() == 651,
    "Keine Seed-Posts im Datensatz":        not comments["comment_id"].isin(
                                                posts.loc[posts["is_seed_post"],
                                                          "post_number"].astype(str)).all(),
    "breadth im Bereich 0–3":               coded["breadth"].dropna().between(0, BREADTH_MAX).all(),
    "valence im Bereich 1–4":               coded["valence"].dropna().between(1, 4).all(),
    "depth im Bereich 0–3":                 coded["depth"].dropna().between(0, 3).all(),
    "Bedingungen balanciert":               final.groupby("version").size().min() > 150,
}
for k, v in checks.items():
    print(f"{'✓' if v else '✗'}  {k}")

---

## Offene Punkte

**1. GerVADER wird `valence` vermutlich nicht allein retten.** Im Test erkennt es idiomatische
Negativität nicht (`„Heisse Luft"`, `„Effektlose Gelaber"` → beide compound = 0). Lexikon-Ansätze
sind stark bei expliziten Sentimentwörtern und schwach bei Ironie und Umschreibung — genau dem,
was in dieser Debatte dominiert. Falls κ auf dem Held-out unter .60 bleibt, in dieser Reihenfolge:

- **Plan B:** `oliverguhr/german-sentiment-bert` — auf deutschen Social-Media-Daten trainiert,
  deutlich stärker als jedes Lexikon, läuft lokal auf CPU. Liefert aber nur pos/neg/neutral,
  *ambivalent* muss weiterhin über eine Regel kommen.
- **Plan C:** Hybrid — VADER-Scores als zusätzliche Features in den LLM-Prompt, oder
  `valence` als einzige Variable weiter manuell codieren (bei 1'211 Kommentaren machbar,
  wenn nur diese eine Variable offen ist).

**2. Referent-Problem.** Codebook-`valence` misst die Bewertung *des 9-Punkte-Plans*, VADER misst
das Sentiment *des Kommentars insgesamt*. Bei „Ich habe Angst vor Anschlägen, der Plan ist gut"
laufen die beiden auseinander. Das ist ein konzeptueller Mismatch, kein Tuning-Problem — gehört
so in die Methods.

**3. Was ich noch von dir brauche:** Pfad/Dateiname des finalen v5-Prompts, die Split-Datei aus
04c (Spaltennamen), und die Bestätigung, welches Modell final ist (qwen3:8b oder llama3.1:8b).
Die Zellen in Abschnitt 0 und 4 sind entsprechend markiert.